In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from json import load
import os

In [4]:
data_dir = r'..\data\raw images'
dir_train1 = os.path.join(data_dir, r'CarDD_release\CarDD_COCO\train\train2017')
dir_train2 = os.path.join(data_dir, r'training')

dir_val1 = os.path.join(data_dir, r'CarDD_release\CarDD_COCO\val\val2017')
dir_val2 = os.path.join(data_dir, r'validation')

In [6]:
with open(r'..\data\instances_train2017.json', encoding='latin-1') as json_data1:
    data = load(json_data1)

labels_DMG_train = {}

for img in data.get("annotations"):
    labels_DMG_train[img.get('image_id')] = [img.get('category_id'), -1]

labels_SV_train = ([[-1, 7]]*452) + ([[-1, 8]]*463) + ([[-1, 9]]*468)

labels_train = list(labels_DMG_train.values()) + labels_SV_train

In [8]:
def get_files_DS1(dir):
    files = []
    for f in os.listdir(dir):
        full_path = os.path.join(dir, f)
        if os.path.isfile(full_path):
            files.append(full_path)
    return files

files_dataset1 = get_files_DS1(dir_train1)
files_dataset3 = get_files_DS1(dir_val1)

In [9]:
def get_files_DS2(dir):
    files_ret = []
    for root, dirs, files in os.walk(dir):
        for file in files:
            files_ret.append(os.path.join(root, file))
    return files_ret

files_dataset2 = get_files_DS2(dir_train2)
files_dataset4 = get_files_DS2(dir_val2)

In [10]:
with open(r'..\data\instances_val2017.json', encoding='latin-1') as json_data2:
    data_val = load(json_data2)

labels_DMG_val = {}

for img in data_val.get("annotations"):
    labels_DMG_val[img.get('image_id')] = [img.get('category_id'), -1]

labels_SV_val = ([[-1, 7]]*82) + ([[-1, 8]]*75) + ([[-1, 9]]*91)

labels_validation = list(labels_DMG_val.values()) + labels_SV_val

In [14]:
files_dataset_train = files_dataset1+files_dataset2
files_dataset_validation = files_dataset3+files_dataset4

train_dataset = tf.data.Dataset.from_tensor_slices((files_dataset_train, labels_train))
val_dataset = tf.data.Dataset.from_tensor_slices((files_dataset_validation, labels_validation))

def load_img_info(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [720, 480])
    img = img / 255.

    label_dmg = label[0]
    label_severity = label[1]

    return img, (label_dmg, label_severity)

train_dataset = train_dataset.map(load_img_info, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.map(load_img_info, num_parallel_calls=tf.data.AUTOTUNE)

train_dataset = train_dataset.shuffle(buffer_size=len(train_dataset))
val_dataset = val_dataset.shuffle(buffer_size=len(val_dataset))

train_dataset = train_dataset.batch(32)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

val_dataset = val_dataset.batch(32)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)